# Integrated Intelligent Systems: Tutorial 1 - Foundations (3 Pts)

This notebook introduces the fundamental components, challenges, and definitions, central to designing and implementing Cognition-Enabled Robot Agents.

## Part 1: The Unified Theoretical Exercise (E1–E6) (1.5 Pt)

**Title:** From Natural Language Request to action execution: The Eva Coffee Scenario

![Alternative Text](coffee-task-photo.png)

### Scenario Background
Eva requests: "Bring me my coffee." Her usual blue mug is in the dishwasher; a white mug is available on the counter. The fill level can vary, and Eva may shift position during execution.

---

### Multi-Question Student Task Sheet (0.25 Pt x 6)
Students work through this unified exercise in the VRB or on paper, structured across the six developmental steps of Lecture 1:

1. **E1 · Opening Judgment (Data vs. Competence):**
   - If an end-to-end vision-language-action model or a demonstration video shows a robot successfully serving coffee 100 times in a row, does that prove the robot "can do the task" across any home environment? State your reasoning and rate your confidence from 1 to 5.
  
     
    **Answer:** No. Confidence: 5/5. A video or 100 consecutive successes only establishes performance on the demonstrated conditions. It does not show that the robot can reliably adapt to other houses, clutter, new cups, full mugs, moving people, or different grasp or friction conditions. True task competence requires success over a defined range of environments, the ability to predict failures, to recognize uncertainty, and to ask for clarification or stop when safety cannot be established. A robust evaluation needs changed-case trials, explicit operating limits, and failure monitoring.
     

2. **E2 · Changed Case Prediction (Physical Bounds):**
   - A half-full white mug was carried successfully along the route. Suppose the mug is now nearly full. Predict the physical consequence during a turn, and identify the untested assumption in the original run.
  
     **Answer:** If the mug is nearly full, there is a higher chance that the coffee will spill when the robot makes a turn. This happens because the liquid moves due to acceleration and there is less space between the coffee and the edge of the mug. The original test only showed that the robot could carry a half-full mug safely. It did not prove that the same speed and turning movement would be safe with a full mug.

     
     

3. **E3 · The Commitment Decision (Warrant Gate):**
   - The blue mug is unavailable and white-mug acceptance is unconfirmed. Choose whether the robot should ask, observe, or proceed. State the single item of evidence that justifies your choice.
  
     
    **Answer:** Choose: ASK.The robot should ask Eva before using the white mug because her acceptance of the substitute is not confirmed. It knows that her preferred blue mug is unavailable, but it cannot assume that she will accept the white one. Therefore, the robot should ask, “Your blue mug is in the dishwasher. Is the white mug okay?” and continue only if Eva agrees. Simply observing that the white mug is available does not confirm her preference.
     



4. **E4 · The Five Questions of Understanding (T1–T5):**
   - Formulate one distinct question for each dimension—Explain ($\text{T1}$), Predict ($\text{T2}$), Design ($\text{T3}$), Bound ($\text{T4}$), and Check ($\text{T5}$)—concerning the coffee delivery episode. At least one question must address a changed condition that a video alone cannot answer.
  
     
    **Answer:** - **T1 – Explain:** Why did the robot choose the white mug instead of the blue one?
               - **T2 – Predict:** What would happen if the mug were almost full and the robot made a sharp turn?
               - **T3 – Design:** How can we design the robot's movement to avoid spilling coffee?
               - **T4 – Bound:** What is the maximum safe speed and turning acceleration when carrying the mug?
               - **T5 – Check:** How can the robot check that the mug is held securely and the coffee has not spilled?
    These are distinct questions, T4 and T2 addresses a changed situation that cannot be established by watching the original demonstration alone.


     

5. **E5 · System Decomposition (Mapping to Architecture):**
   - List the core functional modules required to handle Eva's request (interpreting, perceiving, proposing, predicting, warranting, acting, and recording) and map them onto the Integration Diagram.
  
     
     **Answer:** To complete Eva's coffee delivery request, the robot uses seven main functional modules:
                 **Interpreting:** Understand Eva's request and identify her preferences and the task requirements.
                 **Perceiving:** Detect the available mugs, Eva's location, and possible obstacles using sensors.
                 **Proposing:** Generate a possible plan for picking up and delivering the coffee.
                 **Predicting:** Estimate possible risks, such as spilling coffee, losing grip, or colliding with objects.
                 **Warranting:** Check whether the proposed action is safe, possible, and supported by enough information.
                 **Acting:** Execute the approved plan while monitoring the environment.
                 **Recording:** Save the robot's decisions, predictions, and results for future analysis and improvement.
Integration Diagram:

**Eva's Request → Interpretation → Perception → Proposer → Digital Twin → Warrant Gate → Execution → Recording**

These modules work together to help the robot complete the task safely. If the Warrant Gate rejects a proposed action, the robot should ask for clarification or create a new plan instead of executing an unsafe action.


     

6. **E6 · Transfer & Exit Ticket (Justified Adaptation):**
   - Suppose Eva stands up while the robot is carrying the coffee. Revise the action, predict the potential failure mechanism, and state the exact evidence criterion required before the robot commits to the revised path.
  
     
     **Answer:** **Revised action:** If Eva stands up, the robot should slow down or stop, update her position, and plan a new safe route. It should not continue following the original path. **Potential failure:** A sudden turn or stop could cause the coffee to spill. Also, if the robot does not update Eva's position, it might collide with her. **Evidence required:** Before continuing, the robot must confirm Eva's current position, check that the new path is clear of obstacles, and ensure that the movement will not cause spilling or collisions. The robot should continue only if all safety checks pass. Otherwise, it should wait or plan another route. These checks should also continue during movement.

## Part 2: Interactive Virtual Lab & Python Tutorial (1.5 Pt)

**Title:** Implementing the Embodied Cognitive Architecture

### Scenario Background
Eva requests: "Bring me my coffee."

---

In [5]:
from dataclasses import dataclass
from typing import Dict, Any, Tuple, Optional, List
from collections import Counter
import random
import time

random.seed(42)  # Reproducible classroom demonstration


@dataclass(frozen=True)
class GoalSpecification:
    target_object: str
    destination: str
    target_volume_ml: float
    max_allowable_force_n: float
    max_slosh_wave_mm: float
    max_nav_velocity_mps: float
    time_frame_seconds: float


@dataclass
class ParameterPacket:
    grasp_pose: Tuple[float, float, float]
    max_velocity_mps: float
    tilt_rate_radps: float
    source_generator: str = "stochastic_sampler"


class Proposer:
    """System 1: Generate candidate motion parameters using Gaussian sampling."""
    def propose(self, aggressive=False) -> ParameterPacket:
        if aggressive:
            velocity, tilt = 0.35, 0.01
        else:
            velocity = min(0.30, max(0.05, random.gauss(0.12, 0.03)))
            tilt = min(0.05, max(0.001, random.gauss(0.012, 0.004)))
        return ParameterPacket((0.45, 0.12, 0.30), velocity, tilt)


@dataclass
class Observation:
    timestamp: float
    measured_temperature_c: float
    measured_friction: float
    eva_position: str
    room_busyness: str


@dataclass
class BeliefState:
    pouch_present: bool = True
    pouch_material: str = "flexible_plastic"
    fluid_viscosity_pas: float = 1.2
    vial_ready: bool = True
    camera_active: bool = True
    fallback_path_exists: bool = True
    audit_logger_enabled: bool = True
    ambient_temperature_c: float = 21.5
    surface_friction_coefficient: float = 0.45
    eva_position: str = "living_room"
    room_busyness: str = "medium"
    preferred_cup: str = "blue_ceramic_mug"
    battery_level_pct: float = 100.0


class InterpretationSystem:
    def __init__(self, knowledge_base: Dict[str, Any]):
        self.kb = knowledge_base

    def interpret(self, raw_utterance: str, belief: BeliefState) -> GoalSpecification:
        cup_type = belief.preferred_cup
        cup_properties = self.kb["cups"].get(cup_type, {"max_force": 5.0, "max_wave": 2.0})
        factor = self.kb["room_busyness_limits"].get(belief.room_busyness, 0.5)
        urgent = "schnell" in raw_utterance.lower() or "fast" in raw_utterance.lower()
        goal = GoalSpecification(
            target_object=cup_type,
            destination=belief.eva_position,
            target_volume_ml=12.0 if urgent else 15.0,
            max_allowable_force_n=cup_properties["max_force"],
            max_slosh_wave_mm=cup_properties["max_wave"],
            max_nav_velocity_mps=1.2 * factor,
            time_frame_seconds=30.0 if urgent else 60.0,
        )
        print(f"Goal: {goal.target_object}, max force {goal.max_allowable_force_n} N, "
              f"max speed {goal.max_nav_velocity_mps:.2f} m/s")
        return goal


class SensingSystem:
    def sense(self, true_ambient_temp: float, true_friction: float,
              eva_pos: str, busyness: str) -> Observation:
        return Observation(
            timestamp=time.time(),
            measured_temperature_c=round(true_ambient_temp + random.uniform(-0.5, 0.5), 2),
            measured_friction=round(true_friction + random.uniform(-0.05, 0.05), 2),
            eva_position=eva_pos,
            room_busyness=busyness,
        )


class PerceptionSystem:
    def __init__(self, belief_state: BeliefState):
        self.belief_state = belief_state

    def update(self, observation: Observation) -> None:
        self.belief_state.ambient_temperature_c = observation.measured_temperature_c
        self.belief_state.surface_friction_coefficient = observation.measured_friction
        self.belief_state.eva_position = observation.eva_position
        self.belief_state.room_busyness = observation.room_busyness


class SemanticDigitalTwin:
    def __init__(self, belief_state: BeliefState):
        self.belief = belief_state

    def predict_properties(self, params: ParameterPacket, goal: GoalSpecification) -> Dict[str, float]:
        # Task 2.1: A simplified contact model. Contact time of 0.1 s is assumed.
        contact_depth = params.max_velocity_mps * 0.1
        stiffness = 12.5 * (1.0 + self.belief.surface_friction_coefficient * 0.1)
        if contact_depth > 0.02:
            stiffness *= 1.8 ** (contact_depth / 0.02)
        predicted_force = stiffness * params.max_velocity_mps ** 1.5

        # Task 3.1: Temperature-dependent fluid viscosity.
        temperature = self.belief.ambient_temperature_c
        effective_viscosity = self.belief.fluid_viscosity_pas * max(
            0.05, 1.0 - (temperature - 20.0) * 0.01
        )
        wave_height_mm = (params.tilt_rate_radps * 0.05 / effective_viscosity) * 1000.0

        # A little model error represents the sim-to-real gap.
        return {
            "predicted_force_n": round(max(0.0, predicted_force + random.uniform(-0.2, 0.3)), 2),
            "predicted_wave_mm": round(max(0.0, wave_height_mm + random.uniform(-0.1, 0.2)), 2),
            "predicted_nav_velocity": min(params.max_velocity_mps, goal.max_nav_velocity_mps),
        }


class WarrantGate:
    def __init__(self, sdt: SemanticDigitalTwin):
        self.sdt = sdt

    def validate(self, params: ParameterPacket, goal: GoalSpecification,
                 belief: BeliefState) -> Dict[str, Any]:
        predictions = self.sdt.predict_properties(params, goal)
        # Simplified illustrative percentage consumption, not physical battery data.
        drain_pct = 3.0 + 10.0 * params.max_velocity_mps + 4.0 * params.tilt_rate_radps
        checks = {
            "grounded": belief.vial_ready and belief.pouch_present,
            "feasible": params.max_velocity_mps > 0 and params.tilt_rate_radps > 0,
            "safe": (
                predictions["predicted_force_n"] <= goal.max_allowable_force_n
                and predictions["predicted_wave_mm"] <= goal.max_slosh_wave_mm
                and params.max_velocity_mps <= goal.max_nav_velocity_mps
            ),
            "monitorable": belief.camera_active,
            "repairable": belief.fallback_path_exists,
            "recordable": belief.audit_logger_enabled,
            "energetically_viable": (
                drain_pct < 15.0 and belief.battery_level_pct >= drain_pct + 2.0
            ),
        }
        return {
            "passed": all(checks.values()),
            "failed_criterion": next((k for k, value in checks.items() if not value), None),
            "predictions": predictions,
            "warrant_checks": checks,
            "estimated_battery_drain_pct": round(drain_pct, 2),
        }


class ExecutionEngine:
    def __init__(self):
        self.audit_trail: List[Dict[str, Any]] = []

    def execute_action(self, params: ParameterPacket, goal: GoalSpecification,
                       warrant_result: Dict[str, Any]) -> Dict[str, Any]:
        if not warrant_result["passed"]:
            raise ValueError("Cannot execute a proposal rejected by the Warrant Gate")
        predictions = warrant_result["predictions"]
        actual_force = predictions["predicted_force_n"] + random.uniform(-0.3, 0.6)
        actual_wave = predictions["predicted_wave_mm"] + random.uniform(-0.2, 0.5)
        spill = (actual_force > goal.max_allowable_force_n or
                 actual_wave > goal.max_slosh_wave_mm)
        return {
            "actual_force_n": round(actual_force, 2),
            "actual_wave_mm": round(actual_wave, 2),
            "actual_volume_transferred_ml": goal.target_volume_ml - (3.0 if spill else 0.0),
            "spill_occurred": spill,
            "status": "FAILURE" if spill else "SUCCESS",
        }

    def record_trace(self, asked_utterance: str, goal: GoalSpecification,
                     belief: BeliefState, proposal: ParameterPacket,
                     warrant_result: Dict[str, Any], observation: Observation,
                     committed: bool,
                     execution_outcome: Optional[Dict[str, Any]] = None) -> None:
        self.audit_trail.append({
            "timestamp": time.time(),
            "asked": {
                "raw_utterance": asked_utterance,
                "interpreter_output": {
                    "target": goal.target_object,
                    "max_force_limit_n": goal.max_allowable_force_n,
                    "max_nav_velocity": goal.max_nav_velocity_mps,
                },
            },
            "belief": {
                "eva_position": belief.eva_position,
                "room_busyness": belief.room_busyness,
                "ambient_temp_c": belief.ambient_temperature_c,
                "surface_friction": belief.surface_friction_coefficient,
                "battery_level_pct": belief.battery_level_pct,
            },
            "observed": {
                "timestamp": observation.timestamp,
                "measured_temp_c": observation.measured_temperature_c,
                "measured_friction": observation.measured_friction,
                "eva_position": observation.eva_position,
                "room_busyness": observation.room_busyness,
            },
            "committed": {
                "action_approved": committed,
                "proposed_parameters": {
                    "max_velocity_mps": proposal.max_velocity_mps,
                    "tilt_rate_radps": proposal.tilt_rate_radps,
                },
                "warrant_verdict": warrant_result,
            },
            "execution_outcome": execution_outcome,
        })


def audit_analyzer(audit_trail):
    """Task 5.1: Summarize approvals, rejections and force prediction errors."""
    approved = sum(entry["committed"]["action_approved"] for entry in audit_trail)
    rejected = len(audit_trail) - approved
    reasons = Counter()
    force_errors = []
    for entry in audit_trail:
        verdict = entry["committed"]["warrant_verdict"]
        if not entry["committed"]["action_approved"]:
            reasons.update(k for k, passed in verdict["warrant_checks"].items() if not passed)
        outcome = entry["execution_outcome"]
        if outcome is not None:
            force_errors.append(
                outcome["actual_force_n"] - verdict["predictions"]["predicted_force_n"]
            )
    mean_error = sum(force_errors) / len(force_errors) if force_errors else None
    variance = (sum((e - mean_error) ** 2 for e in force_errors) / len(force_errors)
                if force_errors else None)
    mse = (sum(e ** 2 for e in force_errors) / len(force_errors)
           if force_errors else None)
    results = {
        "approved": approved,
        "rejected": rejected,
        "most_common_rejection": reasons.most_common(1)[0][0] if reasons else None,
        "force_error_variance": variance,
        "force_mean_squared_error": mse,
    }
    print("Audit summary:", results)
    return results


# ------------------- Main demonstration -------------------
knowledge_base = {
    "cups": {
        "blue_ceramic_mug": {"max_force": 4.5, "max_wave": 1.5},
        "plastic_cup": {"max_force": 7.0, "max_wave": 3.0},
    },
    "room_busyness_limits": {"low": 1.0, "medium": 0.6, "high": 0.3},
}

belief_state = BeliefState()
sensing = SensingSystem()
perception = PerceptionSystem(belief_state)
interpreter = InterpretationSystem(knowledge_base)
sdt = SemanticDigitalTwin(belief_state)
warrant_gate = WarrantGate(sdt)
executive = ExecutionEngine()
proposer = Proposer()

print("=== STEP 1: Sensing and Perception ===")
raw_observation = sensing.sense(21.5, 0.45, "living_room", "medium")
perception.update(raw_observation)
print("Observation:", raw_observation)

print("\n=== STEP 2: Interpretation ===")
raw_command = "Bring me my coffee fast!"
goal = interpreter.interpret(raw_command, belief_state)

print("\n=== STEP 3-4: Aggressive proposal and Warrant Gate ===")
proposal_v1 = proposer.propose(aggressive=True)
warrant_v1 = warrant_gate.validate(proposal_v1, goal, belief_state)
print("Proposal:", proposal_v1)
print("Predictions:", warrant_v1["predictions"])
print("Warrant checks:", warrant_v1["warrant_checks"])
print("Gate verdict:", "APPROVED" if warrant_v1["passed"] else "REJECTED")
executive.record_trace(raw_command, goal, belief_state, proposal_v1,
                       warrant_v1, raw_observation, committed=warrant_v1["passed"])
assert not warrant_v1["passed"] and not warrant_v1["warrant_checks"]["safe"]

print("\n=== STEP 5-6: Safer proposal and Execution ===")
# Sample a candidate, then choose conservative values for a clear demonstration.
proposal_v2 = proposer.propose()
proposal_v2.max_velocity_mps = 0.15
proposal_v2.tilt_rate_radps = 0.01
warrant_v2 = warrant_gate.validate(proposal_v2, goal, belief_state)
print("Proposal:", proposal_v2)
print("Predictions:", warrant_v2["predictions"])
print("Warrant checks:", warrant_v2["warrant_checks"])
print("Gate verdict:", "APPROVED" if warrant_v2["passed"] else "REJECTED")
if warrant_v2["passed"]:
    outcome = executive.execute_action(proposal_v2, goal, warrant_v2)
    print("Execution outcome:", outcome)
else:
    outcome = None
    print("No execution: proposal was rejected.")
executive.record_trace(raw_command, goal, belief_state, proposal_v2,
                       warrant_v2, raw_observation,
                       committed=warrant_v2["passed"], execution_outcome=outcome)
assert warrant_v2["passed"]

print("\n=== Audit trail ===")
for i, entry in enumerate(executive.audit_trail, 1):
    print(f"Entry {i}: approved={entry['committed']['action_approved']}, "
          f"outcome={entry['execution_outcome']}")

print("\n=== TASK 2.2: Aggressive 0.35 m/s test ===")
print("Predicted force:", warrant_v1["predictions"]["predicted_force_n"], "N")
print("Safe:", warrant_v1["warrant_checks"]["safe"])

print("\n=== TASK 3.2: Temperature test ===")
test_motion = ParameterPacket((0.45, 0.12, 0.30), 0.10, 0.15)
for temperature in [20.0, 28.0]:
    belief_state.ambient_temperature_c = temperature
    result = warrant_gate.validate(test_motion, goal, belief_state)
    print(f"{temperature:.0f} °C: wave={result['predictions']['predicted_wave_mm']} mm, "
          f"safe={result['warrant_checks']['safe']}")

print("\n=== TASK 4.2: Low battery test ===")
belief_state.ambient_temperature_c = 20.0
belief_state.battery_level_pct = 3.0
battery_motion = ParameterPacket((0.45, 0.12, 0.30), 0.10, 0.01)
battery_result = warrant_gate.validate(battery_motion, goal, belief_state)
print("Battery:", belief_state.battery_level_pct, "%")
print("Estimated drain:", battery_result["estimated_battery_drain_pct"], "%")
print("Energy check:", battery_result["warrant_checks"]["energetically_viable"])
print("Action:", "RECHARGE" if not battery_result["warrant_checks"]["energetically_viable"]
      else "DELIVER" if battery_result["passed"] else "REPLAN")
assert not battery_result["warrant_checks"]["energetically_viable"]

print("\n=== TASK 5.1: Audit analysis ===")
summary = audit_analyzer(executive.audit_trail)
assert summary["approved"] == 1 and summary["rejected"] == 1
print("\nAll demonstration assertions passed.")

=== STEP 1: Sensing and Perception ===
Observation: Observation(timestamp=1793717477.7207706, measured_temperature_c=21.64, measured_friction=0.4, eva_position='living_room', room_busyness='medium')

=== STEP 2: Interpretation ===
Goal: blue_ceramic_mug, max force 4.5 N, max speed 0.72 m/s

=== STEP 3-4: Aggressive proposal and Warrant Gate ===
Proposal: ParameterPacket(grasp_pose=(0.45, 0.12, 0.3), max_velocity_mps=0.35, tilt_rate_radps=0.01, source_generator='stochastic_sampler')
Predictions: {'predicted_force_n': 7.47, 'predicted_wave_mm': 0.39, 'predicted_nav_velocity': 0.35}
Warrant checks: {'grounded': True, 'feasible': True, 'safe': False, 'monitorable': True, 'repairable': True, 'recordable': True, 'energetically_viable': True}
Gate verdict: REJECTED

=== STEP 5-6: Safer proposal and Execution ===
Proposal: ParameterPacket(grasp_pose=(0.45, 0.12, 0.3), max_velocity_mps=0.15, tilt_rate_radps=0.01, source_generator='stochastic_sampler')
Predictions: {'predicted_force_n': 1.0, 'pr

## Questions (0.25 Pt x 4 + 0.5 Pt)

### Task 1: Architectural Mapping & Code Decomposition
* **Question 1.1:** Map the classes and methods implemented in your Python script directly onto the functional blocks of the lecture integration diagram. Explain how data flows between these components during a single cognitive decision cycle.

The Python code represents the main components of the robot's cognitive architecture.

InterpretationSystem: Understands Eva's request and creates a specific goal.

SensingSystem: Collects information from the environment.

PerceptionSystem: Updates the robot's internal belief using sensor data.

Proposer: Generates possible movement parameters.

SemanticDigitalTwin: Predicts the physical effects of the proposed movement.

WarrantGate: Checks whether the proposed action meets the safety requirements.

ExecutionEngine: Executes approved actions and records their results.

During one decision cycle, the robot first collects sensor information and understands the request. It then proposes an action, predicts its outcome, and checks whether it is safe. If the Warrant Gate approves the action, the robot executes it and records the results. Otherwise, it generates a new proposal.


* **Question 1.2:** Complete the proposer class based on a probabilistic model of your choice (e.g., uniform, gaussian, mixture). Justify your choice.

I used a Gaussian distribution to generate movement parameters. This model produces values around a selected average while allowing small variations.

I chose this distribution because it allows the robot to explore different movement speeds and tilt rates without generating completely random values. I also added minimum and maximum limits to keep the parameters within reasonable ranges.

The generated parameters are then evaluated by the Warrant Gate before execution.
  

### Task 2: Non-Linear Contact Mechanics and Material Compliance
In the base script, the contact force on the flexible pouch is modeled using a simplified power-law equation. 
* **Question 2.1:** Modify the `SemanticDigitalTwin.predict_properties` method to account for a dual-stage material compliance curve. If the contact depth exceeds $0.02 \, \text{m}$, the stiffness coefficient should increase exponentially by a factor of $1.8$ due to bottoming out against a rigid internal fixture.

I modified the contact model to represent two stages of material stiffness.
When the contact depth is below 0.02 m, the model uses normal stiffness. When it exceeds 0.02 m, the stiffness increases exponentially because the flexible pouch reaches the rigid internal fixture.
For this simulation, I estimated contact depth using the approach velocity and an assumed contact time. This helps the robot predict higher contact forces during aggressive movements.
  
* **Question 2.2:** Write a test case where the Proposer generates an aggressive approach velocity ($0.35 \, \text{m/s}$). Calculate the predicted force and verify that the Warrant Gate successfully intercepts and rejects the parameter packet before physical execution.

I tested the Proposer using an aggressive approach velocity of 0.35 m/s.
The simulation predicted a contact force of approximately 7.5 N, which exceeds the maximum allowed force of 4.5 N.
As a result, the Warrant Gate rejected the proposed movement before execution. This demonstrates how the safety check prevents potentially dangerous actions.

### Task 3: Advanced Fluid Sloshing and Viscosity Temperature Coupling
Fluids in laboratory automation often exhibit temperature-dependent viscosity changes.
* **Question 3.1:** Enhance the fluid dynamics model so that ambient temperature fluctuations directly alter the effective fluid viscosity using an Arrhenius-type approximation or a linear thermal degradation coefficient ($1\% \, \text{drop per} \, ^\circ\text{C}$ above $20^\circ\text{C}$).

  I modified the fluid model using a linear temperature coefficient. The viscosity decreases by 1% for every degree Celsius above 20°C.
  As temperature increases, the fluid becomes less viscous and can move more easily. This can increase the wave height during movement and create a higher risk of spilling.
  
* **Question 3.2:** Simulate an environment where the ambient temperature rises to $28^\circ\text{C}$. How does this change the predicted wave height for a tilt rate of $0.15 \, \text{rad/s}$? Does the Warrant Gate trigger a safety violation?

  I compared the predicted wave height at 20°C and 28°C using a tilt rate of 0.15 rad/s.
At 20°C, the predicted wave height was approximately 6.25 mm. At 28°C, it increased to around 6.79 mm because the fluid viscosity decreased.
The Warrant Gate detected a safety violation because the predicted wave height exceeded the maximum allowed value of 1.5 mm. The movement was unsafe at both temperatures, but the higher temperature increased the risk.

### Task 4: Expanding the Warrant Gate Checklist
The current Warrant Gate evaluates six core criteria (*grounded, feasible, safe, monitorable, repairable, recordable*). 
* **Question 4.1:** Add a seventh criterion called **`energetically_viable`** to the Warrant Gate dictionary checks. It must evaluate whether the estimated battery drain for the requested trajectory stays below $15\%$ of total capacity.

I added a seventh criterion called `energetically_viable` to the Warrant Gate.
This criterion checks whether the estimated battery consumption is below 15% of the total capacity. It also verifies that the robot has enough remaining battery to complete the movement while keeping a small reserve.
If the energy requirements are not satisfied, the Warrant Gate rejects the proposed action.

* **Question 4.2:** Introduce a low-battery state into the `BeliefState` and demonstrate how a failed `energetically_viable` check redirects the system workflow to request a recharge action rather than proceeding with the coffee delivery.

I tested the system with a battery level of 3%.
The estimated battery consumption was 4.04%, which is higher than the available battery level. Therefore, the `energetically_viable` check returned `False`.
Instead of continuing the coffee delivery, the system requested a recharge action. This helps prevent the robot from running out of energy during execution.

### Task 5: Post-Hoc Epistemic Audit Analysis
A core pillar of trustworthy robotics is the ability to audit *why* a decision was made or rejected.
* **Question 5.1:** Write a post-processing function (`audit_analyzer(audit_trail)`) that parses the recorded episodic log. It should compute and print:
  1. The total number of rejected proposals versus approved executions.
  2. The most frequent warrant criterion responsible for rejection.
  3. The average variance between predicted forces and observed physical outcomes.

I implemented an `audit_analyzer` function to evaluate the robot's recorded decisions.
The function counts approved and rejected proposals and identifies the most common reason for rejection. It also calculates the differences between predicted contact forces and observed physical forces.
This analysis helps evaluate the accuracy of the simulation and identify possible improvements to the robot's decision-making process.

* **Question 5.2:** Discuss in your lab report: *Why does maintaining separate records for `asked`, `belief`, `committed`, and `observed` solve the epistemological gap that pure data-driven imitation learning cannot address?*

Maintaining separate records helps us understand how and why the robot makes decisions.

- **Asked:** Records Eva's original request.
- **Belief:** Represents the robot's understanding of the environment.
- **Committed:** Records the proposed action and whether it was approved.
- **Observed:** Stores information received from the sensors.

By comparing these records with the execution results, we can identify whether an error came from perception, planning, or prediction.

Unlike pure imitation learning, which mainly learns from examples, this approach makes the robot's decisions easier to explain and evaluate. It helps reduce the epistemological gap between the robot's internal understanding and real-world conditions.

## Part 3: Robot Anatomy (0.5 Pt)

**Title:** Links Vs Joints, Actuators Vs Sensors, Effectors Vs Bases Vs Torsos Vs Heads, Wheels Vs Legs Vs Grippers, Proprioception Vs Exteroception Vs Interoception

![Alternative Text](robot-body.jpg)

### Questions

Identify the key functional parts of a robot based on the above picture and briefly indicate how they support the execution of function in the above architecture.

**Answer**

The robot consists of several functional components that work together to perform tasks.

1. **Robot Arm and Joints:** Allow the robot to move its arm and position objects correctly. They support the execution of planned movements.

2. **Gripper (End Effector):** Enables the robot to grasp, hold, and release objects, such as a coffee mug.

3. **Actuators (Motors):** Generate the movement of the robot's joints and mobile base. They execute the commands approved by the Warrant Gate.

4. **Camera / Vision Sensor:** Detects objects, people, and obstacles. It supports the sensing and perception systems.

5. **Distance Sensor (Laser):** Measures distances to surrounding objects and helps the robot avoid collisions during navigation.

6. **Mobile Base:** Allows the robot to move between locations and transport objects.

7. **Robot Controller:** Processes sensor information, plans actions, checks safety requirements, and sends commands to the actuators.

These components support the cognitive architecture by connecting perception, planning, safety checking, and physical execution. For example, the sensors detect the coffee mug, the system plans a safe movement, and the arm and gripper execute the approved action.

---

# Submission guidelines

- Deadline: 03.11.2026, 4 PM
- Submission as Git repo commits pushed within deadlines (Create a repo with the name **iis-assignment-win26** for the whole semester and invite me as collaborator)
- Put all the answers in the notebook. Any other media related to your answer should be inserted in the notebook as well. The notebook and all related media should then be put into a folder called **Assignment1**. 